# Phase 1 — cross-model linear mapping, vi → en

Pipeline: `token(vi) → vit5 encoder → per-token linear map W → flan decoder → token(en)`.

This notebook is the report of record for the numbers. It reads what the pipeline wrote; it does not
recompute it. Regenerate any artefact with the command printed above the cell.

## 0. Environment as measured

In [1]:
import json, sys, platform
from pathlib import Path
import numpy as np, pandas as pd, torch

ROOT = Path.cwd()
RES = ROOT / "results"
print(platform.platform(), "| python", sys.version.split()[0])
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("vram MB", torch.cuda.get_device_properties(0).total_memory // 2**20 if torch.cuda.is_available() else 0)

Windows-11-10.0.26200-SP0 | python 3.12.10
torch 2.7.1+cu128 cuda True NVIDIA GeForce RTX 3050 Laptop GPU
vram MB 4095


## 1. Corpus — what '8M sentences' actually is

`python -m src.build_pairs`

In [2]:
man = json.loads((ROOT / "data" / "pairs_manifest.json").read_text(encoding="utf-8"))
print("rows kept:", f"{man['rows']:,}")
print("split:", man["split"])
print("rule:", man["split_rule"])
print("vi side:", man["vi_side_rule"])

rows kept: 3,456,741
split: {'train': 3452741, 'val': 2000, 'test': 2000}
rule: ORDER BY md5(vi); first 2000 = test, next 2000 = val, rest = train
vi side: higher count of non-ASCII chars; ties dropped


## 2. Map fitting — ridge sweep

`python -m src.fit_tok shards` then `python -m src.fit_tok fit`

In [3]:
sweep = pd.read_csv(RES / "ridge_sweep.csv")
display(sweep)
m = np.load(ROOT / "data" / "map_vi_en.npz")
print("chosen lambda", float(m["lam"]), "| shards", int(m["shards"]),
      "| pairs", f"{int(m['n_pairs']):,}", "| tokens", f"{int(m['n_tokens']):,}")

,direction,lambda,mse_train,mse_val
0,vi_en,0.0000,7.557588,7.630433
1,vi_en,0.0001,7.557588,7.630433
2,vi_en,0.0010,7.557588,7.630433
3,vi_en,0.0100,7.557588,7.630433
4,vi_en,0.1000,7.557588,7.630433
5,vi_en,1.0000,7.557588,7.630433
6,vi_en,10.0000,7.557588,7.630433
7,en_vi,0.0000,531.468272,535.279213
8,en_vi,0.0001,531.468272,535.279213
9,en_vi,0.0010,531.468272,535.279213


chosen lambda 0.0 | shards 18 | pairs 3,452,741 | tokens 85,089,885


## 3. Geometry, with the controls that make cosine interpretable

`python -m src.eval_geometry`

In [4]:
geo = pd.read_csv(RES / "mapping_metrics.csv")
meta = json.loads((RES / "mapping_metrics.json").read_text(encoding="utf-8"))
display(geo)
print({k: meta[k] for k in ("n_test_pairs", "chance_r1", "protocol")})

,direction,variant,token_cosine,token_mse,sentence_cosine,sentence_mse,retrieval_at_1,margin_mean,margin_frac_positive
0,vi_en,map,0.4515,7.485087,0.7510,1.369905,0.4875,-0.0111,0.4875
1,vi_en,identity,0.0038,687.924500,0.0100,200.040405,0.0010,-0.0944,0.0010
2,vi_en,mean,0.3234,8.463607,0.5759,2.051192,0.0005,-0.1588,0.0005
3,en_vi,map,0.4512,530.446167,0.8144,69.988617,0.4400,-0.0205,0.4400
4,en_vi,identity,0.0038,687.924500,0.0100,200.040405,0.0005,-0.0929,0.0005
5,en_vi,mean,0.3563,587.657654,0.6927,105.241920,0.0005,-0.1459,0.0005


{'n_test_pairs': 2000, 'chance_r1': 0.0005, 'protocol': 'paired retrieval: query = mapped source tokens; gallery = target tokens of the same 2,000 pairs'}


The identity row is the point: without a map the two spaces are nearly orthogonal. The mean row is the
trap: predicting the dataset mean earns cosine for free, which is why retrieval@1 is the number that
carries the claim.

## 4. Generation — three channels, one decoder

`python -m src.train_decoder --pairs 100000` then `python -m src.generate` then `python -m src.score`

In [5]:
met = json.loads((RES / "metrics.json").read_text(encoding="utf-8"))
rows = [{"channel": k, **{kk: vv for kk, vv in v.items() if isinstance(vv, (int, float))}}
        for k, v in met.items() if isinstance(v, dict) and "bleu" in v]
tbl = pd.DataFrame(rows).set_index("channel")
cols = ["bleu", "chrf", "comet", "degenerate_rate", "vi_word_rate", "empty_rate", "mean_hyp_words"]
display(tbl[[c for c in cols if c in tbl.columns]].round(4))
if "paired_native_minus_mapped" in met:
    print(json.dumps(met["paired_native_minus_mapped"], indent=2))

,bleu,chrf,comet,degenerate_rate,vi_word_rate,empty_rate,mean_hyp_words
channel,,,,,,,
mapped,6.2799,28.1463,0.6000,0.1440,0.0001,0.001,20.6471
native,3.8475,18.7216,0.4426,0.2155,0.0010,0.000,21.6575
identity,0.0114,10.9909,0.2879,0.0305,0.0362,0.077,18.7156
baseline,22.2671,45.3089,0.7519,0.0015,0.0007,0.000,15.9395


{
  "mean": -2.4362296299161397,
  "ci95": [
    -3.0251067677293855,
    -1.8176714874285491
  ],
  "p_other_le_mapped": 1.0,
  "n_boot": 1000,
  "reading": "positive mean = this channel beats mapped; P(other<=mapped) near 1 means mapped is significantly better"
}


### 4.1 Examples (one per channel, same Vietnamese source)

In [6]:
gens = {c: [json.loads(l) for l in (RES / f"gen_{c}.jsonl").open(encoding="utf-8")]
        for c in met if (RES / f"gen_{c}.jsonl").exists()}
i = 0
print("VI :", gens["mapped"][i]["vi"])
print("REF:", gens["mapped"][i]["ref"])
for c, rows in gens.items():
    print(f"{c:9s}:", rows[i]["hyp"])

VI : Tôi đang tập trung nghiên cứu sự phát triển nhận thức ở trẻ vị thành niên.
REF: Get their feedback on your questionnaire by asking the following questions: Was the questionnaire easy to understand?
mapped   : I'm trying to focus on developing cognitive skills in children.
native   : I'm not sure what the hell I'm doing here.
identity : van appetiteomni naissance left change trigger grace titleened new blow Rrunvoir relief right non seriesborne appetite header new),,back partial trigger whiteunLAND appetite uprightnous new generic Roy triggerner revival reduced claw wound joint resurrectionner entitlement non,root wrong partial header right disorder opener token, rep sortiener gall borrowed new RER trigger new header wrong revival front prescription upright revival partialrien larva right new
baseline : I'm focused on researching cognitive development in teenagers.


## 5. Food probe on the frozen flan encoder (§3.1)

`python -m src.probe`

In [7]:
if (RES / "probe_metrics.json").exists():
    display(pd.Series(json.loads((RES / "probe_metrics.json").read_text(encoding="utf-8"))))
else:
    print("probe not run")

n                    {'train': 99929, 'val': 21413, 'test': 21414}
classes          {'1': 'RecipeNLG title+ingredients', '0': 'MTE...
weight_decay                                                   1.0
val_auc                                                   0.999999
test_auc                                                       1.0
test_accuracy                                              0.99986
test_f1                                                     0.9999
embedding        flan-t5-base encoder, masked-mean, max_len 64,...
dtype: object

## 6. Data track (M1)

In [8]:
dw = ROOT / "data-work"
for p in sorted(dw.glob("structured/*")):
    print(f"{p.relative_to(ROOT)}  {p.stat().st_size/1024:.0f} KB")
for p in sorted(dw.glob("markdown/*.md")):
    print(f"{p.relative_to(ROOT)}  {p.stat().st_size/1024:.0f} KB")

data-work\structured\cache  160 KB
data-work\structured\cost.json  1 KB
data-work\structured\culture_facts.jsonl  552 KB
data-work\structured\facts.db  804 KB
data-work\structured\facts_stats.json  1 KB
data-work\structured\failures_dishes.json  0 KB
data-work\structured\failures_facts.json  96 KB
data-work\structured\REPORT.md  15 KB
data-work\structured\vn_dishes.csv  204 KB
data-work\markdown\us_american_cuisine.md  30 KB
data-work\markdown\us_food_younger_land.md  60 KB
data-work\markdown\us_sioux_chef.md  2 KB
data-work\markdown\vn_mon_an_truyen_thong.md  291 KB
data-work\markdown\vn_van_hoa_am_thuc.md  450 KB


## 7. What this does not show

* en → vi is measured only geometrically; generating Vietnamese needs the vit5 decoder.
* The decoder is base-size and trained for one epoch on 100k pairs; the BLEU ceiling is set by that,
  not by the map. The claim is the *gap between channels*, not the absolute BLEU.
* COMET is reported only if `python -m src.score --comet` was run; the da reference-free variant
  requires the English source, so be explicit about which COMET checkpoint produced the number.